# EDA

## *EDA五大环节*

1.数据加载与基础信息探查

2.数据清洗（缺失值、重复值、异常值）

3.单变量分析（单列独立分布，分数值 / 分类）

4.多变量交叉分析（特征之间关联、特征与目标变量关系）

5.数据结论总结

## *1.数据加载与基础信息探查*

#### 一、数据集整体规模信息

- **样本总量与特征数量**：`df.shape` 快速判断量级（小/中/大数据），预判分析计算量与建模难度；区分宽表（特征多）与长表（样本多）
- **索引与主键**：索引是否有序、是否乱序，是否存在唯一主键标识列（如 Row ID、Order ID）

#### 二、字段类型与数据结构信息

- **类型分类**：`df.dtypes` 区分数值型（int/float）、分类型（object/category）、时间型（datetime），预判后续分析路径
  - 数值型 → 直方图、箱线图
  - 分类型 → 频次柱状图
  - 时间型 → 时序趋势
- **识别无用字段**：ID、编号、备注、全空文本等无业务价值列，后续可直接删除
- **明确目标变量**：区分自变量 / 因变量，锁定建模目标

#### 三、缺失值初步情况

- `df.info()` 查看各字段缺失数量与占比，快速定位缺失严重字段，提前预判处理方向（删除 / 填充 / 插值）
- 区分随机零星缺失 vs 大面积全空字段，为数据清洗做铺垫

#### 四、数值字段基础统计分布

- `df.describe()` 获取：
  - **集中趋势**：均值、中位数
  - **离散程度**：标准差
  - **区间范围**：最大值、最小值、四分位数
- 快速发现明显异常信号：最小值负数、最大值远超业务合理区间、均值与中位数差距过大（数据严重偏态）

#### 五、分类字段基础分布

- **唯一值数量**（`df.nunique()`）：区分高基数（手机号、地址）与低基数分类（性别、等级）
- **各类别占比**（`value_counts()`）：判断是否存在类别极度不平衡（如 99% 样本集中在单一类别）

#### 六、业务层面隐藏问题

- **字段命名规范**：中英文混杂、空格/特殊符号 → 后续重命名清洗
- **数据格式错乱**：数值存储为字符串、日期是文本格式 → 类型转换
- **冗余列识别**：重复含义特征 → 提前筛选精简，避免多重共线性

#### 总结：探查核心目标

> 不用深入画图，仅靠基础函数（`shape`、`info`、`describe`、`nunique`、`value_counts`）快速**定位数据隐患**，明确数据结构，为第二步数据清洗和后续可视化分析提供处理依据。

## 2.数据清洗（缺失值、重复值、异常值）

### 2.1 缺失值

#### 1. 先判断缺失类型（决定处理逻辑）

- **MCAR**（完全随机缺失）、**MAR**（ 条件随机缺失）、**MNAR**（非随机缺失，最难处理）
- 缺失是**随机零散分布**，还是**集中在某段时间、某类人群、高/低数值区间**——集中缺失往往携带业务信号

#### 2. 缺失程度

- 单字段缺失占比：极低（< 5%）→ 可删行 / 中等 → 需填充策略 / 超 50% → **直接删字段**
- 区分是**样本整行大面积缺失**（可能是采集链路故障），还是**仅个别字段零星缺失**

#### 3. 缺失产生原因（业务根源）

- 采集故障、接口漏传、用户未填写、业务逻辑本身无数据、脏数据
- 缺失本身可能携带业务信息（如"未填收入" ≈ 低意愿披露），不能随便填充抹掉信号
- 建议：新增 `_is_missing` 标记列保留"是否缺失"这一信息

#### 4. 数据与模型限制

- **线性模型**对填充值敏感（影响系数估计）；**树模型**容错更高（依赖分裂点）
- ⚠️ **不能引入数据泄露**：训练集的填充规则（均值、KNN 参数等）必须仅从训练集学习，不能混入测试集信息

#### 5. 填充方案的优缺点对比

| 方案 | 优点 | 缺点 |
|------|------|------|
| **删除** | 简单直接 | 丢样本，尾部/小众数据损失严重 |
| **均值/中位数** | 简单快速 | 抹平分布、扭曲长尾、低估方差 |
| **众数（分类）** | 保留最常见类别 | 少数类别被淹没 |
| **前向/后向填充** | 适合时序数据 | 不适合无序数据 |
| **模型填充（KNN/迭代/MICE）** | 精准，利用多列关联 | 计算量大，小样本易过拟合 |
| **新增缺失标记列** | 保留"是否缺失"的业务信息 | 增加特征维度 |

#### 6. 业务可解释性

- 填充后数值是否符合现实逻辑？会不会出现不合理的数值（如负金额、年龄 > 150）？
- MNAR 场景下简单填充（均值/0）会严重失真，直接误导结论——需结合领域知识或单独建模缺失机制

#### 7. 线上落地一致性

- 离线训练阶段的填充规则、阈值、模型参数，**线上推理时必须完全复用**，不能有任何偏差
- 建议将填充逻辑封装为可序列化的 pipeline（如 `sklearn` 的 `ColumnTransformer` + `SimpleImputer`），确保训练/推理一致

#### 补充：用热力图判断缺失类型（MCAR / MAR / MNAR）

缺失值热力图（`sns.heatmap` / `missingno.matrix`）是判断缺失模式的**第一眼工具**。

| 热力图特征 | 对应缺失类型 | 含义 |
|------------|-------------|------|
| 缺失点**均匀散落**，无规律 | **MCAR**（完全随机缺失） | 纯随机，最简单，删行/简单填充均可 |
| 字段 A 的缺失**集中出现在**字段 B 的某个取值区间 | **MAR**（条件随机缺失） | 缺失可由其他已观测变量解释，可用模型填充 |
| 缺失**集中在字段自身的高/低值区间**，与其他字段无关 | **MNAR**（非随机缺失） | 缺失与未观测到的值本身有关，最难处理 |

**实操步骤：**

1. **`missingno.matrix` 全局矩阵图**
   ```python
   import missingno as msno
   msno.matrix(df.sample(500))  # 抽样看，大数据集全量太密
   ```
   - 右边 sparkline 显示每行缺失计数，若大量行集中在高缺失区域 → 可能是采集批次故障
   - 空白竖条 = 某字段大面积缺失；空白横条 = 某样本行大面积缺失

2. **`missingno.heatmap` 字段间缺失相关性热力图** — 判断 MAR 的关键证据
   ```python
   msno.heatmap(df)
   ```
   - 相关系数接近 **+1**：字段 A 缺失 → 字段 B 也缺失（联动缺失，同一采集源故障）
   - 相关系数接近 **-1**：字段 A 缺失 → 字段 B 一定有值（互斥缺失）
   - 相关系数接近 **0**：字段间缺失互不相关 → 倾向 MCAR

3. **缺失 vs 非缺失分组对比** — 验证 MAR 假设
   ```python
   df['col_A_missing'] = df['col_A'].isnull()
   df.groupby('col_A_missing')['col_B'].describe()
   ```
   - 两组在 `col_B` 上分布显著不同 → col_A 缺失与 col_B 有关 → MAR
   - 两组分布几乎一致 → 更可能是 MCAR

**判断决策树：**
```
热力图缺失均匀散落？ ──是──→ MCAR → 简单填充/删行
    │否
    ├── 缺失与另一字段取值有关？ ──是──→ MAR → 用该字段信息做模型填充
    │否
    └── 缺失与字段自身有关？ ──是──→ MNAR → 业务介入或单独建模
```

### 2.2 重复值

### 2.3 异常值

>箱线图由下最小值、下四分位、中位、上四分位、上最大值组成，其中超过上下最值的点为异常值。

>长尾分布核心特征：绝大多数样本集中在少量类别 / 数值区间，极少数类别 / 区间拥有极多样本，整体呈现 “头重尾长”,在箱线图上最常见右长尾（正偏，绝大多数业务数据：收入、订单、销量、时长）
>- 箱体整体靠左，中位数贴近下四分位数 Q1；
>- 上须很短，大量离散圆点（离群点） 向右延伸很远，拉出一条长长的尾巴；
>- 下须很短，左侧几乎没有异常点。

**长尾类别**
- 按偏移方向
    - 右长尾
    - 左长尾
- 按业务数据类型
    - 连续数值长尾
    - 类别长尾

**数据存在长尾分布需要考虑的全维度问题**
- 业务层面：先搞懂长尾成因
    - 二八现象
    - 长尾是否有业务价值
        - 尾部有价值：小众品类、下沉用户、冷门需求，不能直接丢弃；
        - 尾部无价值：噪声、测试数据、无效脏数据，可过滤。
    - 长尾是天然分布还是脏数据导致
- 数据分析与可视化层面
    - 常规图表会失真
        - 直方图、折线图头部柱子过高，尾部几乎看不见，无法观察尾部细节；
解决：对数坐标轴、分箱、截断视图、帕累托曲线。
    - 箱线图只能看单侧离群点，无法量化长尾严重程度
    - 均值指标失效
- 特征工程
    - 数值特征（金额、时长、频次）
        - 非线性变换压缩长尾
        - 分箱离散化
        - 截断 / 缩尾（Winsorize）
        - 归一化标准化失效
    - 分类 / 类别长尾（样本不均衡，分类任务灾难）
- 模型训练层面的影响
    - 线性模型极度敏感
    - 树模型相对耐受，但仍有隐患
    - 梯度不稳定、收敛变慢
    - 预测偏差
- 数据清洗与样本筛选



**何时长尾是可以忽略的，何是又是必须处理的**
- **长尾可以忽略的场景**
    - 业务目标只关心头部群体，尾部无价值
    - 使用对极值鲁棒的统计 / 模型，长尾不会产生干扰
        - 仅做描述分析，只用中位数、分位数、MAE，不用均值、MSE
        - 只用树模型（XGB/LGB/RF）做回归，未使用线性回归、逻辑回归
树模型天然按区间切分，不会被极端值拉扯梯度，轻微长尾不用额外变换。
    - 长尾是少量无关脏数据，可直接过滤删除
    - 尾部样本数量极少，且不参与核心评估
    - 仅做粗略趋势分析，不需要精准预测
- **长尾必须处理的场景**
    - 尾部是业务核心关注对象（最关键）
    - 使用线性模型、深度学习回归、最小二乘损失
    - 依赖均值、MSE、RMSE 作为核心评价指标
    - 多分类任务，类别长尾（样本不均衡）


## 3.单变量分析（单列独立分布，分数值 / 分类）

### 3.1 全变量类型汇总

单变量分析覆盖 **8 大类变量**，每类有各自的分析目的、方法和图表。收到数据后先按此体系分类，再逐类探查。

---

#### 1. 连续数值变量

**定义：** 可无限细分、能做加减乘除运算的数字，存在小数/区间范围。如销售额、利润、运费、单价、时长。

**分析目的：** 掌握集中区间与离散程度；判断分布形态（正态/偏态、单峰/双峰）；识别极端异常值；判断是否需要特征变换。

**分析方法：** `describe()` 基础统计；直方图 + KDE + 箱线图可视化；极端分位数截断避免图表被压缩。

**典型图表：** 直方图 + KDE、箱线图 → *详见 3.2~3.4 节*

---

#### 2. 无序分类变量

**定义：** 固定类别，类别间无大小先后顺序。如地区、产品大类、运输方式、客户细分。

**分析目的：** 统计各类样本量与占比；判断类别是否均衡，识别头部主流 vs 小众冷门；统计类别总数，区分低基数/高基数特征。

**分析方法：** `value_counts()` 频次统计 + 百分比；高基数字段合并尾部小众类别。

**典型图表：** 类别柱状图、分层饼图

---

#### 3. 有序等级变量

**定义：** 分类自带天然顺序，但不能直接数值运算。如订单优先级（低/中/高）、客户星级 1~5、评价等级。

**分析目的：** 观察样本集中在低档/中档/高档；判断是否两极分化；后续建模可编码为有序数字。

**分析方法：** 频次统计，**绘图严格按等级顺序排列**，不可乱序。

**典型图表：** 有序柱状图（类别顺序固定）

---

#### 4. 布尔二值变量

**定义：** 仅两种取值：0/1、是/否。如是否亏损、是否退货、是否包邮。

**分析目的：** 判断样本是否严重失衡（一类占 99%）；失衡数据会干扰建模，提前标记。

**分析方法：** 统计两类数量与占比。

**典型图表：** 简易柱状图、双区块饼图

---

#### 5. 时间变量

**定义：** 日期/时间格式数据，可拆解年/季/月/日。如订单日期、发货日期。

**分析目的：** 确定数据完整时间跨度；挖掘年度增长趋势、月度淡旺季波动；观察折扣、订单量、利润随时间同步变化规律。

**分析方法：** 拆分年/月/年月维度；按月/年聚合销售额、订单量、利润等指标。

**典型图表：** 月度折线面积图、月度盈亏柱状图、年度分组柱状图、双轴时序图

---

#### 6. 长文本变量

**定义：** 大段文字内容。如客户评论、备注、反馈文案。

**分析目的：** 查看缺失比例、文本长度分布；提取高频关键词，初步感知文本整体情绪。

**分析方法：** 文本长度统计、词频统计。

**典型图表：** 文本长度分布直方图、词云图

---

#### 7. ID 唯一标识变量

**定义：** 每条样本独一无二的编号。如订单 ID、客户 ID、商品编号。

**分析目的：** **仅校验是否存在重复 ID**，无业务分布分析价值，一般直接排除出可视化。

**分析方法：** `duplicated()` 查重。

**无可视化图表。**

---

#### 8. 地理坐标变量

**定义：** 经纬度纯数字，本质属于连续数值，单独归类便于区分。

**分析目的：** 查看坐标分布区间，识别缺失、异常坐标。

**分析方法：** 同连续数值分析；多变量阶段再做地图可视化。

**典型图表：** KDE、箱线图（单变量阶段）

---

#### 快速分类总览

| 大类 | 包含类型 | 核心分析方式 |
|------|---------|-------------|
| 数值类 | 连续数值、经纬度 | 直方图 + KDE + 箱线图 |
| 分类类 | 无序分类、有序等级、布尔二值 | 频次统计 + 柱状图 |
| 时序类 | 时间日期 | 时序聚合 + 折线/柱状图 |
| 文本类 | 长文本 | 长度分布 + 词频/词云 |
| 标识类 | ID 主键 | 仅查重，不做分布分析 |

### 3.5 补充：KDE 详解（带宽调节与实操要点）

> 3.2 节已覆盖 KDE 的读图方法，此处补充技术细节。

#### 一、KDE 是什么

可以理解成**顺滑无分段的直方图**。普通直方图把数字切成固定区间画柱子，区间切法一变，图形观感差别很大；KDE 不做分段，把所有数据点融合成一条连续平滑曲线，曲线高低代表对应数值样本的密集程度——本质是对总体概率密度函数的模拟。

#### 二、适用 / 不适用场景

| 适合 | 不适合 |
|------|--------|
| 所有数值型单列分布探索 | 分类文本字段 |
| 搭配分类字段分组，对比不同类别下同一数值的分布差异 | 离散数字且取值极少（星级 1-5、性别 0/1、等级）→ 用柱状图 |

#### 三、使用时注意要点 — 带宽（bandwidth）

唯一关键调节项是**带宽**：

- **带宽太小** → 曲线凹凸杂乱，把随机噪声当成真实规律
- **带宽太大** → 曲线过度平缓，掩盖双峰、分层等关键特征
- **日常单变量探查**直接使用工具默认自动带宽即可，无需手动调整
- ⚠️ **强偏态数据陷阱**：`gaussian_kde` 基于全量数据标准差估算带宽，极端值会把带宽撑大，KDE 被压成一条平线。正确做法是只用 P1~P99 范围内的数据估计 KDE（详见 eda.ipynb 3.1 代码）

#### 四、对整套数据分析流程的帮助

1. **承接数据清洗**：曲线尾部稀疏长尾区域，大概率存在异常值，辅助筛选异常数据
2. **完成单变量分析核心任务**：完整摸清每一列数值的分布特征
3. **为多变量分析铺垫**：识别人群分层、分布偏移，提前规划交叉分析思路
4. **为后续建模做前置判断**：根据分布形态确定是否需要标准化、对数转换、分箱等特征操作

## 4.多变量交叉分析（特征之间关联、特征与目标变量关系）

## 5.数据结论总结